# 1.2 Web search

A chat model's weights are frozen at **training time**. It cannot look up last week's news unless you give it a **tool** that hits the live web.

This notebook:

1. Asks an agent (no tools) how current its knowledge is.
2. Wraps [Tavily](https://tavily.com/) as a LangChain tool and calls it directly.
3. Attaches that tool to the agent and asks a time-sensitive question.

Requires `TAVILY_API_KEY` and `OPENAI_API_KEY` in `.env`.

## Setup

Load API keys from `.env`. Run this cell first.

In [ ]:
from dotenv import load_dotenv

# Puts OPENAI_API_KEY, TAVILY_API_KEY, etc. into os.environ
load_dotenv()

## Without web search

`create_agent` with only a model — no `tools`. The reply to "How up to date is your training knowledge?" will describe a cutoff, not live retrieval.

In [ ]:
from langchain.agents import create_agent

# No tools: answers come only from the model's trained weights
agent = create_agent(
    model="gpt-5-nano"
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
# Last message is the model's self-description of its cutoff
print(response['messages'][-1].content)

## Add a web search tool

`TavilyClient` talks to Tavily's search API. Decorating `web_search` with `@tool` exposes it to the agent: name, docstring, and `query: str` become the schema the model fills in.

The last line **invokes the tool directly** (no agent) so you can see raw search JSON before wiring it up.

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

# Uses TAVILY_API_KEY from the environment
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

# Smoke-test the tool with a question that changes over time
web_search.invoke("Who is the current mayor of San Francisco?")

Pass `tools=[web_search]` so the agent can call Tavily when the question needs current facts. Expect a tool-call message in the trace, then a final answer grounded in the search results.

In [ ]:
# Same model, now allowed to search
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

### Inspect the message list

Pretty-print `response['messages']`: human question → AI `tool_calls` → tool result → final AI answer.

If LangSmith tracing is on, you can also open a saved run (example public trace below).

In [ ]:
from pprint import pprint

pprint(response['messages'])

Example LangSmith trace: [public run](https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r)